In [0]:
# ===================================================
# BLOCK 1 — IMPORTS AND MODEL CONFIGURATION (PYTHON)
# ===================================================

"""
Define governed sources, targets, timezone handling and deterministic key
generation used across every conformed dimension.
"""

from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

CATALOG = "semiconplus_portfolio"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

LOCAL_TIMEZONE = "Asia/Taipei"
PRODUCTION_DAY_START_HOUR = 8

SILVER_PRODUCTION_LOTS = f"{CATALOG}.{SILVER_SCHEMA}.production_lots"
SILVER_UNIT_TEST_RESULTS = f"{CATALOG}.{SILVER_SCHEMA}.unit_test_results"
SILVER_SITES = f"{CATALOG}.{SILVER_SCHEMA}.sites"
SILVER_PRODUCT_GROUPS = f"{CATALOG}.{SILVER_SCHEMA}.product_groups"
SILVER_DEVICES = f"{CATALOG}.{SILVER_SCHEMA}.devices"
SILVER_EQUIPMENT = f"{CATALOG}.{SILVER_SCHEMA}.equipment"

DIM_DATE = f"{CATALOG}.{GOLD_SCHEMA}.dim_date"
DIM_SITE = f"{CATALOG}.{GOLD_SCHEMA}.dim_site"
DIM_PRODUCT_GROUP = f"{CATALOG}.{GOLD_SCHEMA}.dim_product_group"
DIM_DEVICE = f"{CATALOG}.{GOLD_SCHEMA}.dim_device"
DIM_EQUIPMENT = f"{CATALOG}.{GOLD_SCHEMA}.dim_equipment"
LOT_IDENTIFIER_MAPPING = f"{CATALOG}.{GOLD_SCHEMA}.lot_identifier_mapping"
DIM_LOT = f"{CATALOG}.{GOLD_SCHEMA}.dim_lot"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")
spark.conf.set("spark.sql.session.timeZone", "UTC")


def stable_key(entity_name, business_key_column):
    """Return a deterministic SHA-256 surrogate key."""

    return F.sha2(
        F.concat_ws(
            "|",
            F.lit(entity_name.upper()),
            F.coalesce(
                business_key_column.cast("string"),
                F.lit("<NULL>"),
            ),
        ),
        256,
    )


print("Day 2 model configuration loaded.")

In [0]:
# ===================================================
# BLOCK 2 — BUILD THE COMPLETE DATE DIMENSION (PYTHON)
# ===================================================

"""
Create one date row for every production-lot business date. The source maximum
date, rather than the current calendar date, identifies the latest completed
production date in this controlled historical dataset.
"""

date_bounds = (
    spark.table(SILVER_PRODUCTION_LOTS)
    .agg(
        F.min("production_date").alias("minimum_date"),
        F.max("production_date").alias("maximum_date"),
    )
    .first()
)

minimum_date = date_bounds["minimum_date"]
maximum_date = date_bounds["maximum_date"]

assert minimum_date is not None
assert maximum_date is not None
assert minimum_date <= maximum_date

date_df = (
    spark.sql(
        f"""
        SELECT EXPLODE(
            SEQUENCE(
                TO_DATE('{minimum_date}'),
                TO_DATE('{maximum_date}'),
                INTERVAL 1 DAY
            )
        ) AS full_date
        """
    )
    .select(
        F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
        "full_date",
        F.year("full_date").alias("calendar_year"),
        F.quarter("full_date").alias("calendar_quarter_number"),
        F.concat(F.lit("Q"), F.quarter("full_date")).alias("calendar_quarter"),
        F.month("full_date").alias("calendar_month_number"),
        F.date_format("full_date", "MMMM").alias("calendar_month_name"),
        F.date_format("full_date", "yyyy-MM").alias("year_month"),
        (
            F.year("full_date") * F.lit(100)
            + F.month("full_date")
        ).alias("year_month_sort"),
        F.concat(
            F.year("full_date"),
            F.lit("-Q"),
            F.quarter("full_date"),
        ).alias("year_quarter"),
        (
            F.year("full_date") * F.lit(10)
            + F.quarter("full_date")
        ).alias("year_quarter_sort"),
        F.weekofyear("full_date").alias("iso_week_number"),
        F.dayofmonth("full_date").alias("day_of_month"),
        F.dayofweek("full_date").alias("day_of_week_number"),
        F.date_format("full_date", "EEEE").alias("day_of_week_name"),
        (F.col("full_date") == F.lit(maximum_date)).alias(
            "is_latest_complete_production_date"
        ),
        F.current_timestamp().alias("_gold_processed_at_utc"),
    )
)

(
    date_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_DATE)
)

print(
    f"Date dimension created: {minimum_date} through {maximum_date}; "
    f"{date_df.count():,} rows."
)

In [0]:
# ===================================================
# BLOCK 3 — BUILD SITE AND PRODUCT-GROUP DIMENSIONS (PYTHON)
# ===================================================

"""
Publish small reference dimensions with deterministic keys so reruns do not
change relationship values in downstream facts or Power BI.
"""

site_df = (
    spark.table(SILVER_SITES)
    .select(
        stable_key("SITE", F.col("site_id")).alias("site_key"),
        F.trim("site_id").alias("site_id"),
        F.trim("site_name").alias("site_name"),
        F.trim("country").alias("country"),
        F.trim("timezone").alias("timezone"),
        F.lit(True).alias("active_flag"),
        F.current_timestamp().alias("_gold_processed_at_utc"),
    )
)

product_group_df = (
    spark.table(SILVER_PRODUCT_GROUPS)
    .select(
        stable_key(
            "PRODUCT_GROUP",
            F.col("product_group_id"),
        ).alias("product_group_key"),
        F.trim("product_group_id").alias("product_group_id"),
        F.trim("product_group_name").alias("product_group_name"),
        F.trim("business_unit").alias("business_unit"),
        F.col("restricted").cast("boolean").alias("restricted_flag"),
        F.lit(True).alias("active_flag"),
        F.current_timestamp().alias("_gold_processed_at_utc"),
    )
)

for target_table, target_df in [
    (DIM_SITE, site_df),
    (DIM_PRODUCT_GROUP, product_group_df),
]:
    (
        target_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

print(f"Sites published: {site_df.count():,}")
print(f"Product groups published: {product_group_df.count():,}")

In [0]:
# ===================================================
# BLOCK 4 — BUILD DEVICE AND EQUIPMENT DIMENSIONS (PYTHON)
# ===================================================

"""
Resolve device-to-product-group and equipment-to-site relationships while
retaining deterministic surrogate keys and operational attributes.
"""

device_df = (
    spark.table(SILVER_DEVICES).alias("device")
    .join(
        spark.table(DIM_PRODUCT_GROUP).alias("product_group"),
        F.col("device.product_group_id")
        == F.col("product_group.product_group_id"),
        "left",
    )
    .select(
        stable_key("DEVICE", F.col("device.device_id")).alias("device_key"),
        F.trim("device.device_id").alias("device_id"),
        F.trim("device.device_name").alias("device_name"),
        F.col("product_group.product_group_key"),
        F.trim("device.product_group_id").alias("product_group_id"),
        F.trim("device.package_type").alias("package_type"),
        F.col("device.target_yield").cast("double").alias("target_yield"),
        F.trim("device.lifecycle_status").alias("lifecycle_status"),
        F.lit(True).alias("active_flag"),
        F.current_timestamp().alias("_gold_processed_at_utc"),
    )
)

equipment_df = (
    spark.table(SILVER_EQUIPMENT).alias("equipment")
    .join(
        spark.table(DIM_SITE).alias("site"),
        F.col("equipment.site_id") == F.col("site.site_id"),
        "left",
    )
    .select(
        stable_key(
            "EQUIPMENT",
            F.col("equipment.equipment_id"),
        ).alias("equipment_key"),
        F.trim("equipment.equipment_id").alias("equipment_id"),
        F.col("site.site_key"),
        F.trim("equipment.site_id").alias("site_id"),
        F.trim("equipment.equipment_model").alias("equipment_model"),
        F.trim("equipment.equipment_type").alias("equipment_type"),
        F.col("equipment.rated_units_per_hour")
        .cast("int")
        .alias("rated_units_per_hour"),
        F.lit(True).alias("active_flag"),
        F.current_timestamp().alias("_gold_processed_at_utc"),
    )
)

for target_table, target_df in [
    (DIM_DEVICE, device_df),
    (DIM_EQUIPMENT, equipment_df),
]:
    (
        target_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

print(f"Devices published: {device_df.count():,}")
print(f"Equipment published: {equipment_df.count():,}")


In [0]:
# ===================================================
# BLOCK 5 — DERIVE LOT COMPLETION AND PRODUCTION DAY (PYTHON)
# ===================================================

"""
Use the latest sampled unit-test timestamp as the controlled completion proxy.
Fall back to the lot start only when no unit-test timestamp is available.
"""

unit_test_completion_df = (
    spark.table(SILVER_UNIT_TEST_RESULTS)
    .groupBy("lot_id")
    .agg(
        F.max("event_timestamp_utc").alias(
            "maximum_unit_test_timestamp_utc"
        )
    )
)

lot_mapping_candidates_df = (
    spark.table(SILVER_PRODUCTION_LOTS).alias("lot")
    .join(
        unit_test_completion_df.alias("test"),
        F.col("lot.lot_id") == F.col("test.lot_id"),
        "left",
    )
    .select(
        F.trim("lot.lot_id").alias("source_lot_id"),
        F.upper(F.trim("lot.device_id")).alias("device_id"),
        F.coalesce(
            F.col("test.maximum_unit_test_timestamp_utc"),
            F.col("lot.start_timestamp_utc"),
        ).alias("lot_completion_timestamp_utc"),
    )
    .withColumn(
        "lot_completion_timestamp_local",
        F.from_utc_timestamp(
            "lot_completion_timestamp_utc",
            LOCAL_TIMEZONE,
        ),
    )
    .withColumn(
        "production_date",
        F.to_date(
            F.expr(
                "lot_completion_timestamp_local "
                f"- INTERVAL {PRODUCTION_DAY_START_HOUR} HOURS"
            )
        ),
    )
    .withColumn(
        "test_batch_id",
        F.concat(
            F.date_format("production_date", "yy"),
            F.col("device_id"),
            F.date_format("production_date", "MMdd"),
        ),
    )
)

invalid_candidate_count = lot_mapping_candidates_df.filter(
    F.col("source_lot_id").isNull()
    | F.col("device_id").isNull()
    | F.col("lot_completion_timestamp_utc").isNull()
    | F.col("production_date").isNull()
    | F.col("test_batch_id").isNull()
).count()

assert invalid_candidate_count == 0, (
    f"Invalid lot-mapping candidates: {invalid_candidate_count}"
)

print(f"Lot mapping candidates: {lot_mapping_candidates_df.count():,}")
display(lot_mapping_candidates_df.orderBy("source_lot_id").limit(20))

In [0]:
# ===================================================
# BLOCK 6 — CREATE THE PERSISTENT LOT-MAPPING TABLE (PYTHON)
# ===================================================

"""
Create the mapping table once. Later runs insert only previously unseen source
lots, preserving all existing simulated identifiers and sequence numbers.
"""

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {LOT_IDENTIFIER_MAPPING} (
        source_lot_id STRING NOT NULL,
        test_batch_id STRING NOT NULL,
        test_lot_sequence INT NOT NULL,
        test_lot_id STRING NOT NULL,
        production_date DATE NOT NULL,
        lot_completion_timestamp_utc TIMESTAMP NOT NULL,
        lot_completion_timestamp_local TIMESTAMP NOT NULL,
        mapping_created_at_utc TIMESTAMP NOT NULL,
        simulated_hierarchy_flag BOOLEAN NOT NULL
    )
    USING DELTA
    COMMENT 'Persistent simulated test-batch and test-lot identifier mapping'
    """
)

mapping_count_before = spark.table(LOT_IDENTIFIER_MAPPING).count()
print(f"Persistent mappings before MERGE: {mapping_count_before:,}")

In [0]:
# ===================================================
# BLOCK 7 — ASSIGN IDENTIFIERS TO NEW LOTS (PYTHON)
# ===================================================

"""
Assign new sequences after the maximum existing sequence for each test batch.
Completion timestamp and source-lot ID provide deterministic within-run order.
"""

existing_mapping_df = spark.table(LOT_IDENTIFIER_MAPPING)

existing_max_sequence_df = (
    existing_mapping_df
    .groupBy("test_batch_id")
    .agg(
        F.max("test_lot_sequence").alias(
            "existing_max_test_lot_sequence"
        )
    )
)

new_lot_candidates_df = (
    lot_mapping_candidates_df.alias("candidate")
    .join(
        existing_mapping_df.select("source_lot_id").alias("existing"),
        F.col("candidate.source_lot_id")
        == F.col("existing.source_lot_id"),
        "left_anti",
    )
)

new_lot_order_window = Window.partitionBy("test_batch_id").orderBy(
    "lot_completion_timestamp_utc",
    "source_lot_id",
)

new_mapping_rows_df = (
    new_lot_candidates_df
    .join(existing_max_sequence_df, "test_batch_id", "left")
    .withColumn(
        "test_lot_sequence",
        F.coalesce(
            F.col("existing_max_test_lot_sequence"),
            F.lit(0),
        )
        + F.row_number().over(new_lot_order_window),
    )
    .withColumn(
        "test_lot_id",
        F.concat(
            F.col("test_batch_id"),
            F.lit("T"),
            F.lpad(F.col("test_lot_sequence").cast("string"), 3, "0"),
        ),
    )
    .select(
        "source_lot_id",
        "test_batch_id",
        F.col("test_lot_sequence").cast("int"),
        "test_lot_id",
        "production_date",
        "lot_completion_timestamp_utc",
        "lot_completion_timestamp_local",
        F.current_timestamp().alias("mapping_created_at_utc"),
        F.lit(True).alias("simulated_hierarchy_flag"),
    )
)

new_mapping_count = new_mapping_rows_df.count()

print(f"New mappings prepared: {new_mapping_count:,}")
display(new_mapping_rows_df.orderBy("test_lot_id").limit(20))

In [0]:
# ===================================================
# BLOCK 8 — MERGE NEW LOT IDENTIFIERS (PYTHON)
# ===================================================

"""
Insert new mappings without updating existing rows. This is the core stability
control that prevents historical test-lot identifiers from being renumbered.
"""

new_mapping_rows_df.createOrReplaceTempView("new_lot_identifier_mappings")

spark.sql(
    f"""
    MERGE INTO {LOT_IDENTIFIER_MAPPING} AS target
    USING new_lot_identifier_mappings AS source
      ON target.source_lot_id = source.source_lot_id
    WHEN NOT MATCHED THEN INSERT (
        source_lot_id,
        test_batch_id,
        test_lot_sequence,
        test_lot_id,
        production_date,
        lot_completion_timestamp_utc,
        lot_completion_timestamp_local,
        mapping_created_at_utc,
        simulated_hierarchy_flag
    ) VALUES (
        source.source_lot_id,
        source.test_batch_id,
        source.test_lot_sequence,
        source.test_lot_id,
        source.production_date,
        source.lot_completion_timestamp_utc,
        source.lot_completion_timestamp_local,
        source.mapping_created_at_utc,
        source.simulated_hierarchy_flag
    )
    """
)

mapping_count_after = spark.table(LOT_IDENTIFIER_MAPPING).count()

assert mapping_count_after == mapping_count_before + new_mapping_count, (
    "Persistent lot-mapping reconciliation failed."
)

print(f"Persistent mappings after MERGE: {mapping_count_after:,}")
print(f"Mappings inserted by this run: {new_mapping_count:,}")

In [0]:
# ===================================================
# BLOCK 9 — BUILD THE CONFORMED LOT DIMENSION (PYTHON)
# ===================================================

"""
Combine source lot attributes, persistent simulated identifiers and conformed
dimension keys into one row per source lot.
"""

lot_df = (
    spark.table(SILVER_PRODUCTION_LOTS).alias("lot")
    .join(
        spark.table(LOT_IDENTIFIER_MAPPING).alias("mapping"),
        F.col("lot.lot_id") == F.col("mapping.source_lot_id"),
        "inner",
    )
    .join(
        spark.table(DIM_DEVICE).alias("device"),
        F.col("lot.device_id") == F.col("device.device_id"),
        "left",
    )
    .join(
        spark.table(DIM_PRODUCT_GROUP).alias("product_group"),
        F.col("lot.product_group_id")
        == F.col("product_group.product_group_id"),
        "left",
    )
    .join(
        spark.table(DIM_SITE).alias("site"),
        F.col("lot.site_id") == F.col("site.site_id"),
        "left",
    )
    .join(
        spark.table(DIM_EQUIPMENT).alias("equipment"),
        F.col("lot.equipment_id") == F.col("equipment.equipment_id"),
        "left",
    )
    .select(
        stable_key("LOT", F.col("lot.lot_id")).alias("lot_key"),
        F.col("lot.lot_id").alias("source_lot_id"),
        "mapping.test_batch_id",
        "mapping.test_lot_id",
        "mapping.test_lot_sequence",
        "device.device_key",
        "product_group.product_group_key",
        "site.site_key",
        "equipment.equipment_key",
        F.col("lot.device_id"),
        F.col("lot.product_group_id"),
        F.col("lot.site_id"),
        F.col("lot.equipment_id"),
        F.col("lot.start_timestamp_utc").alias("lot_start_timestamp_utc"),
        "mapping.lot_completion_timestamp_utc",
        "mapping.lot_completion_timestamp_local",
        "mapping.production_date",
        F.year("mapping.production_date").alias("production_year"),
        "mapping.mapping_created_at_utc",
        "mapping.simulated_hierarchy_flag",
        F.lit(True).alias("active_flag"),
        F.current_timestamp().alias("_gold_processed_at_utc"),
    )
)

(
    lot_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_LOT)
)

print(f"Lot dimension published: {lot_df.count():,}")
display(lot_df.orderBy("test_lot_id").limit(20))

In [0]:
# ===================================================
# BLOCK 10 — DIMENSION BUILD RESULT (PYTHON)
# ===================================================

dimension_results = [
    (DIM_DATE, spark.table(DIM_DATE).count()),
    (DIM_SITE, spark.table(DIM_SITE).count()),
    (DIM_PRODUCT_GROUP, spark.table(DIM_PRODUCT_GROUP).count()),
    (DIM_DEVICE, spark.table(DIM_DEVICE).count()),
    (DIM_EQUIPMENT, spark.table(DIM_EQUIPMENT).count()),
    (LOT_IDENTIFIER_MAPPING, spark.table(LOT_IDENTIFIER_MAPPING).count()),
    (DIM_LOT, spark.table(DIM_LOT).count()),
]

display(
    spark.createDataFrame(
        dimension_results,
        ["table_name", "row_count"],
    )
)

print("DAY 2 CONFORMED DIMENSION BUILD: COMPLETED")
print(
    "Run this notebook a second time. On the second run, "
    "'Mappings inserted by this run' must equal 0."
)